# 09 — Thermodynamic consistency and energy release

This notebook covers one property that is easy to lose without noticing, and
one result that surprises most users of rate libraries.

A reaction network relaxes to the equilibrium implied by the **ratios of the
rates it is given**. The equilibrium solver computes a composition from nuclear
masses and partition functions. If those two disagree, a network calculation
and an NSE calculation performed on the same data will settle in different
places — and nothing in either calculation will warn you.

In [ ]:
# The notebooks run against the installed package.  If you are working from a
# checkout without installing, uncomment the two lines below.
# import sys, pathlib
# sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nucnetpy as nn

print("nucnetpy version:", nn.__version__)

## Where the inconsistency comes from

Rate libraries such as JINA ReacLib supply the forward and reverse directions
of a reaction as **separately fitted** expressions. Each fit is good, but their
ratio is not exactly the equilibrium constant implied by the masses shipped
alongside them.

Below is a small network with a forward rate and a library-style reverse rate.
`consistent_reverse_network` pairs each reaction with its inverse, keeps the
exothermic member, and regenerates the other from detailed balance.

In [ ]:
def demo_network():
    net = nn.Network()
    for name, z, a, me in [("he4", 2, 4, 2.425), ("c12", 6, 12, 0.0),
                           ("o16", 8, 16, -4.737), ("ne20", 10, 20, -7.042)]:
        net.add_species(nn.Species(name, z=z, a=a, mass_excess=me, spin=0.0))
    for i, (x, y, p) in enumerate([("c12", "he4", "o16"),
                                   ("o16", "he4", "ne20")]):
        net.reactions.add(nn.Reaction.from_names(
            [x, y], [p, "gamma"], constant_rate=1.0e2, q_value=5.0))
        net.reactions.add(nn.Reaction.from_names(
            [p, "gamma"], [x, y], constant_rate=2.0e1 * (1 + 0.5 * i),
            q_value=-5.0))
    return net

library = demo_network()
consistent = nn.consistent_reverse_network(library)
print("reactions:", len(library.reactions.reactions),
      "->", len(consistent.reactions.reactions))

A detailed-balance reverse rate carries a factor
$\exp(-Q/kT)$ and so depends strongly on temperature. A constant is simply the
wrong shape for it.

In [ ]:
reverse = [r for r in consistent.reactions.reactions
           if r.source == "detailed_balance"][0]
print("reverse of:", reverse.string)
pd.DataFrame([{"T9": t9, "library (constant)": 20.0,
               "detailed balance": reverse.bare_rate(t9)}
              for t9 in [1.0, 2.0, 3.0, 4.0, 5.0]])

## The test: net flux at equilibrium

Here is the cleanest statement of the problem. Take the composition that the
equilibrium solver returns, and evaluate the forward and reverse flux of each
reaction there. **At equilibrium every net flux must vanish** — that is what
equilibrium means.

If it does not, the network will drive the composition away from NSE.

In [ ]:
T9, RHO = 4.0, 1.0e7
nse = nn.solve_nse(library, t9=T9, rho=RHO, ye=0.5)
abundances = nse.abundances

print("NSE composition:")
for k, v in sorted(abundances.items(), key=lambda kv: -kv[1]):
    print(f"   {k:6s} X = {library.species[k].a * v:.5f}")

def imbalance(network):
    """Forward and reverse flux of each reaction, at the NSE composition."""
    keys = {r.key: r for r in network.reactions.reactions}
    rows = []
    for r in network.reactions.reactions:
        rev = keys.get((r.key[1], r.key[0]))
        if rev is None or r.q_value < 0:
            continue
        f = r.flux(abundances, t9=T9, rho=RHO)
        b = rev.flux(abundances, t9=T9, rho=RHO)
        rows.append({"reaction": r.string, "forward": f, "reverse": b,
                     "|f-r|/(f+r)": abs(f - b) / max(f + b, 1e-300)})
    return pd.DataFrame(rows)

In [ ]:
# Using the library's own reverse rates.
imbalance(library)

In [ ]:
# Using reverse rates rebuilt from detailed balance.
imbalance(consistent)

With the library's own reverse rates the imbalance is of order
one: the reverse flux is negligible against the forward flux, so the network
would burn straight through the composition that NSE calls equilibrium. Rebuilt
from detailed balance, every net flux vanishes to machine precision.

This is a statement about the rate library, not about either solver.

## What it costs on real data

On a production JINA database the effect is smaller but far from negligible.
Reproduced with `validation/demonstration.py` in the repository, burning
$^{28}$Si at $T_9=5$ and $\rho=10^{8}\,$g cm$^{-3}$ on the alpha chain until
the composition stops changing, then comparing with a separately solved NSE
over the same species set the network can populate:

| reverse rates | median difference | maximum difference |
|---|---|---|
| JINA library fits | $4.5\times10^{-2}$ | $4.8\times10^{-1}$ |
| detailed balance  | $7.0\times10^{-8}$ | $1.6\times10^{-7}$ |

Nearly six orders of magnitude. The library's reverse rates differ from detailed
balance by a median factor of 1.10 and up to 1.72, which is what displaces the
equilibrium abundances by a few per cent.

```bash
python validation/demonstration.py \
    --nuclides nuclides.xml --reactions reaction_data.xml
```

A user who saw the 4.5 per cent agreement and concluded the code was working
would have been right about the code and wrong about the abundances.

Read the two rows differently. The first is an independent comparison: the
network runs on the library's own forward and reverse fits while the NSE solve
is built from the mass table, so the two share no numerical input. The second
is not. `consistent_reverse_network` derives each reverse rate from the same
equilibrium prefactor `solve_nse` uses, so a mistake in the masses, the
$(2J+1)$ factor or the partition-function convention would shift both sides
together and cancel. That figure says the integrator, the stoichiometry and the
equilibrium solver treat one shared formulation consistently; it does not say
that formulation is correct.

The equilibrium must be solved over the species the network can populate. An
alpha chain cannot liberate a nucleon, so it keeps $Y_n=Y_p=0$, while an
unrestricted NSE puts $6.3\times10^{-6}$ of the mass in free nucleons and
depresses everything else to pay for it. Leaving them in the solve and merely
omitting them from the error inflates the second row to
$4.5\times10^{-6}$.

### The trade-off

Deriving reverse rates from mass differences enforces consistency, but it
discards whatever measurement or evaluation went into the library's reverse
fit. Where the library value is better than the masses it would be derived
from, keep the library value. `consistent_reverse_network` is an option, not a
default.

Because $\exp(-Q/kT)$ is far too steep to interpolate, the rebuilt rate is
attached as a function evaluated at whatever temperature the solver requests.
`tabulate=True` samples it onto a grid instead — less accurate, but the result
can be written to XML.

In [ ]:
exact = nn.consistent_reverse_network(library)
tabulated = nn.consistent_reverse_network(library, tabulate=True)

r_exact = [r for r in exact.reactions.reactions if r.source == "detailed_balance"][0]
r_tab = [r for r in tabulated.reactions.reactions if r.source == "detailed_balance"][0]

pd.DataFrame([{"T9": t9, "exact": r_exact.bare_rate(t9),
               "tabulated": r_tab.bare_rate(t9),
               "relative error": abs(r_tab.bare_rate(t9) - r_exact.bare_rate(t9))
                                 / max(r_exact.bare_rate(t9), 1e-300)}
              for t9 in [0.5, 1.0, 2.0, 4.0, 8.0]])

## Energy release

The nuclear energy generation rate follows from the change in the total mass
excess of the composition,

$$\varepsilon = -N_{\rm A}\,C\sum_i \dot Y_i\, \Delta M_i,\qquad C = 1.602176634\times10^{-6}\ {\rm erg\ MeV^{-1}},$$

where $C$ converts the mass excesses from MeV, so that $\varepsilon$ comes out
in erg g$^{-1}$ s$^{-1}$ rather than MeV g$^{-1}$ s$^{-1}$.

It does not use the reaction $Q$-values stored in the rate library. That
matters for the same reason as above: it is internally consistent with the
adopted mass-excess table, the same one the equilibrium solver uses. It may
differ from library $Q$-values when the two data sources use different nuclear
masses.

Neutrino losses are not subtracted and must be accounted for separately.

In [ ]:
from nucnetpy.analysis import nuclear_energy_release, nuclear_energy_generation_rate

burn = nn.Network()
burn.add_species(nn.Species("si28", z=14, a=28, mass_excess=-21.493, spin=0.0))
burn.add_species(nn.Species("ni56", z=28, a=56, mass_excess=-53.907, spin=0.0))

released = nuclear_energy_release(burn, {"si28": 1.0 / 28.0}, {"ni56": 1.0 / 56.0})
q = 2 * (-21.493) - (-53.907)
print(f"28Si -> 56Ni releases {released:.4e} erg/g")
print(f"  check: Q = 2 ME(28Si) - ME(56Ni) = {q:.3f} MeV per 56Ni formed")
print(f"         {q / 56.0 * 6.02214076e23 * 1.602176634e-6:.4e} erg/g")

The demonstration in the manuscript releases
$1.75\times10^{17}\,$erg g$^{-1}$, against this ceiling of
$1.88\times10^{17}$, consistent with its final $X(^{56}$Ni$) = 0.886$.

## Diagnostics

The analysis module carries the quantities used in the NucNet Tools workflows:
per-species timescales $Y_i/|\dot Y_i|$, which identify the stiffest components
of a system; per-reaction contributions to $\dot Y_e$; the entropy generation
rate; the $s$-process neutron exposure; and time-integrated reaction currents.

In [ ]:
from nucnetpy.analysis import system_timescales
import copy

work = copy.deepcopy(consistent)
work.zones = [nn.Zone(abundances={"c12": 0.02, "he4": 0.05, "o16": 0.03},
                      properties={"t9": "4.0", "rho": "1e7"})]
ts = system_timescales(work, 0)
pd.DataFrame(sorted(((k, v) for k, v in ts.items() if np.isfinite(v)),
                    key=lambda kv: kv[1]),
             columns=["species", "timescale Y/|dY/dt| (s)"])